In [1]:
# pip install pathlib typing
import logging
from pathlib import Path
from typing import Any, Dict, Optional
import pandas as pd
import requests
from PIL import Image
import requests
from io import BytesIO


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

## Функция загрузки данных из Open-Meteo API

Функция ниже реализует defensive programming:
- timeout для защиты от зависания
- проверка HTTP-статуса
- проверка схемы ответа
- безопасное преобразование в DataFrame

In [2]:
def fetch_weather_timeseries(latitude: float, longitude: float, days_ahead: int = 3) -> Optional[pd.DataFrame]:
    base_url = "https://api.open-meteo.com/v1/forecast"
    params: Dict[str, Any] = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": days_ahead,
        "timezone": "auto",
    }

    try:
        response = requests.get(base_url, params=params, timeout=10.0)
        response.raise_for_status()
        payload = response.json()
        hourly_node = payload.get("hourly", {})
        if "time" not in hourly_node or "temperature_2m" not in hourly_node:
            logging.error("Ошибка схемы ответа: не найдены ключи 'time' и/или 'temperature_2m'.")
            return None

        times = hourly_node.get("time", [])
        temps = hourly_node.get("temperature_2m", [])

        if len(times) != len(temps):
            logging.error("Неконсистентные данные: длина time и temperature_2m не совпадает.")
            return None

        df = pd.DataFrame({
            "timestamp": times,
            "temperature_celsius": temps,
        })

        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        return df

    except requests.exceptions.HTTPError as http_err:
        logging.error("HTTP ошибка: %s", http_err)
    except requests.exceptions.ConnectionError:
        logging.error("Ошибка соединения: проверьте интернет или доступность API.")
    except requests.exceptions.Timeout:
        logging.error("Таймаут запроса: сервер отвечает слишком долго.")
    except requests.exceptions.RequestException as req_err:
        logging.error("Неожиданная ошибка requests: %s", req_err)
    except ValueError as json_err:
        logging.error("Ошибка парсинга JSON: %s", json_err)

    return None

## Загрузка данных и первичная проверка

Сначала получаем данные, затем валидируем:
- shape
- обязательные колонки
- пропуски

In [3]:
df_weather = fetch_weather_timeseries(latitude=35.6895, longitude=139.6917, days_ahead=3)

if df_weather is None:
    raise RuntimeError("Не удалось получить данные из API.")

required_cols = ["timestamp", "temperature_celsius"]
missing_cols = [c for c in required_cols if c not in df_weather.columns]

print("Shape:", df_weather.shape)
print("Missing columns:", missing_cols)
print("NaN by column:")
print(df_weather[required_cols].isna().sum())

if missing_cols:
    raise ValueError(f"Отсутствуют обязательные колонки: {missing_cols}")

df_weather.head()

Shape: (72, 2)
Missing columns: []
NaN by column:
timestamp              0
temperature_celsius    0
dtype: int64


,timestamp,temperature_celsius
0,2026-04-11 00:00:00,14.7
1,2026-04-11 01:00:00,14.0
2,2026-04-11 02:00:00,13.4
3,2026-04-11 03:00:00,13.4
4,2026-04-11 04:00:00,13.4


In [4]:
summary = df_weather["temperature_celsius"].describe()
summary

count    72.000000
mean     16.198611
std       4.390434
min       8.600000
25%      12.900000
50%      15.100000
75%      19.600000
max      25.400000
Name: temperature_celsius, dtype: float64

## Автоматизация Excel-отчета

Ниже функция формирует готовый к использованию корпоративный отчет:
- форматированная шапка
- автоширина колонок
- freeze panes
- автофильтр

In [5]:
def generate_corporate_excel_report(df: pd.DataFrame, output_filename: str = "weather_report.xlsx") -> Optional[Path]:
    if df is None or df.empty:
        print("DataFrame пустой.")
        return None

    output_path = Path(output_filename).resolve()

    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        sheet_name = "Executive Summary"
        df.to_excel(writer, sheet_name=sheet_name, index=False)

        workbook = writer.book
        worksheet = writer.sheets[sheet_name]

        header_format = workbook.add_format({
            "bold": True,
            "align": "center",
            "valign": "vcenter",
            "fg_color": "#003366",
            "font_color": "#FFFFFF",
            "border": 1,
        })

        for col_idx, col_name in enumerate(df.columns):
            worksheet.write(0, col_idx, col_name, header_format)

            data_max_len = df[col_name].astype(str).str.len().max()
            header_len = len(str(col_name))
            optimal_width = max(data_max_len if pd.notna(data_max_len) else 0, header_len) + 3
            worksheet.set_column(col_idx, col_idx, optimal_width)

        total_rows, total_cols = df.shape
        worksheet.freeze_panes(1, 0)
        worksheet.autofilter(0, 0, total_rows, total_cols - 1)

    print(f"Отчет сохранен: {output_path}")
    return output_path

In [6]:
report_path = generate_corporate_excel_report(df_weather, output_filename="lesson_40_weather_report.xlsx")
report_path

Отчет сохранен: C:\projc\teaching_activities\data_analysis_course\data_analysis_course\lessons\lesson_40_weather_report.xlsx


WindowsPath('C:/projc/teaching_activities/data_analysis_course/data_analysis_course/lessons/lesson_40_weather_report.xlsx')

## Практическое задание

1. Измените координаты на любой другой город и получите новый прогноз.
2. Добавьте в API-запрос дополнительные погодные признаки (например, windspeed).